In [1]:
# Cell 1
import sys
import subprocess
import os
import json
import warnings


def install_requirements():
    required_packages = [
        'pandas',
        'numpy',
        'lightgbm',
        'scikit-learn',
        'joblib',
        'pyarrow',
        'scipy',
        'torch'
    ]

    for package in required_packages:
        import_name = 'sklearn' if package == 'scikit-learn' else package
        try:
            __import__(import_name)
        except ImportError:
            print(f"Package '{package}' not found. Installing")
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', package]
            )

    print('All dependencies installed.')


install_requirements()

import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
import torch
import torch.nn as nn

from scipy.special import expit
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings('ignore', category=FutureWarning)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

All dependencies installed.


In [2]:
# Cell 2
SUPERVISED_MODEL_CANDIDATES = [
    'FinanceRiskManager_winner.pkl',
    'sentinel_x_winner.pkl'
]

PREPROCESSING_ARTIFACT_PATH = 'preprocessing_artifacts.pkl'
FINANCIAL_POLICY_PATH = 'tri_state_cost_report.json'
AE_MODEL_PATH = 'abuse_ring_sentinel_autoencoder.pth'
AE_ARTIFACT_PATH = 'abuse_ring_sentinel_artifacts.pkl'

M6_REAL_PATH = 'test_m6_real.parquet'
M6_STRESS_PATH = 'test_m6_stress.parquet'

FINAL_REPORT_PATH = 'm6_final_evaluation_report.json'
FINAL_PREDICTIONS_PATH = 'm6_final_predictions.csv'
ABLATION_PATH = 'm6_policy_ablation.csv'

PRODUCTION_USE_DUAL_MODEL = True
PRODUCTION_ESCALATION_POLICY = 'graduated'

GRADUATED_MSE_SEVERITY_MULTIPLIER = 1.5
GRADUATED_RISK_MIDPOINT_FRACTION = 0.5

In [3]:
# Cell 3
supervised_model_path = None
for candidate in SUPERVISED_MODEL_CANDIDATES:
    if os.path.exists(candidate):
        supervised_model_path = candidate
        break

required_paths = [
    PREPROCESSING_ARTIFACT_PATH,
    FINANCIAL_POLICY_PATH,
    AE_MODEL_PATH,
    AE_ARTIFACT_PATH,
    M6_REAL_PATH,
    M6_STRESS_PATH
]

if supervised_model_path is None:
    raise FileNotFoundError(
        'No frozen supervised winner found. Checked: '
        + ', '.join(SUPERVISED_MODEL_CANDIDATES)
    )

for path in required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Required file missing: {path}'
        )

lgb_model = joblib.load(
    supervised_model_path
)

prep = joblib.load(
    PREPROCESSING_ARTIFACT_PATH
)

feature_columns = list(
    prep['feature_columns']
)

categorical_features = list(
    prep['categorical_features']
)

category_maps = prep['category_maps']
winning_model_name = prep['winning_model_name']

with open(
    FINANCIAL_POLICY_PATH,
    'r',
    encoding='utf-8'
) as f:
    policy = json.load(f)

thresholds = policy['deployment_thresholds']

t_low = float(
    thresholds['auto_approve_max_risk']
)

t_high = float(
    thresholds['auto_block_min_risk']
)

fin_cfg = policy['financial_assumptions']
review_cfg = policy['review_assumptions']

if not 0 <= t_low <= t_high <= 1:
    raise ValueError(
        'Invalid Phase 2 deployment thresholds.'
    )

if winning_model_name not in {
    'Focal_Loss_LightGBM',
    'Standard_LightGBM'
}:
    raise ValueError(
        f'Unknown winning model name: {winning_model_name}'
    )

print(f'Frozen supervised model: {supervised_model_path}')
print(f'Winning model type: {winning_model_name}')
print(f'Phase 2 thresholds: [{t_low:.6f}, {t_high:.6f}]')

Frozen supervised model: FinanceRiskManager_winner.pkl
Winning model type: Focal_Loss_LightGBM
Phase 2 thresholds: [0.210000, 0.520000]


In [4]:
# Cell 4
class FraudDetector(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dims=(128, 64, 16),
        latent_dim=8,
        dropout_p=0.10
    ):
        super().__init__()

        h1, h2, h3 = hidden_dims

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.LayerNorm(h1),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(h1, h2),
            nn.LayerNorm(h2),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(h2, h3),
            nn.LayerNorm(h3),
            nn.GELU(),
            nn.Linear(h3, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, h3),
            nn.LayerNorm(h3),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(h3, h2),
            nn.LayerNorm(h2),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(h2, h1),
            nn.LayerNorm(h1),
            nn.GELU(),
            nn.Linear(h1, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


ae_checkpoint = torch.load(
    AE_MODEL_PATH,
    map_location=device,
    weights_only=False
)

ae_artifacts = joblib.load(
    AE_ARTIFACT_PATH
)

ae_model = FraudDetector(
    input_dim=int(ae_checkpoint['input_dim']),
    hidden_dims=tuple(ae_checkpoint['hidden_dims']),
    latent_dim=int(ae_checkpoint['latent_dim']),
    dropout_p=float(ae_checkpoint['dropout_p'])
).to(device)

ae_model.load_state_dict(
    ae_checkpoint['model_state_dict']
)
ae_model.eval()

required_ae_keys = [
    'numerical_features',
    'imputation_medians',
    'lower_clip',
    'upper_clip',
    'scaler',
    'anomaly_threshold'
]

missing_ae_keys = [
    key for key in required_ae_keys
    if key not in ae_artifacts
]

if missing_ae_keys:
    raise KeyError(
        f'Autoencoder artifact missing keys: {missing_ae_keys}'
    )

numerical_features = list(
    ae_artifacts['numerical_features']
)

ae_medians = pd.Series(
    ae_artifacts['imputation_medians']
)
ae_lower_clip = pd.Series(
    ae_artifacts['lower_clip']
)
ae_upper_clip = pd.Series(
    ae_artifacts['upper_clip']
)
ae_scaler = ae_artifacts['scaler']
ae_threshold = float(
    ae_artifacts['anomaly_threshold']
)

print(f'Autoencoder input dimension: {ae_checkpoint["input_dim"]}')
print(f'Autoencoder threshold: {ae_threshold:.6f}')

Autoencoder input dimension: 405
Autoencoder threshold: 0.723460


In [5]:
# Cell 5
def add_temporal_features(df):
    df = df.copy()

    if 'TransactionDT' in df.columns:
        df['TransactionHour'] = (
            (df['TransactionDT'] // 3600) % 24
        ).astype(np.int8)

        df['TransactionDay'] = (
            df['TransactionDT'] // (24 * 3600)
        ).astype(np.int32)

        df['HourSin'] = np.sin(
            2 * np.pi * df['TransactionHour'] / 24
        ).astype(np.float32)

        df['HourCos'] = np.cos(
            2 * np.pi * df['TransactionHour'] / 24
        ).astype(np.float32)

        df.drop(
            columns=['TransactionDT'],
            inplace=True
        )

    return df


def prepare_lgb_features(df):
    transformed = add_temporal_features(df)

    missing = [
        col for col in feature_columns
        if col not in transformed.columns
    ]

    if missing:
        raise ValueError(
            f'LightGBM features missing: {missing}'
        )

    X = transformed[
        feature_columns
    ].copy()

    for col in categorical_features:
        categories = pd.Index(
            category_maps[col]
        )
        X[col] = pd.Categorical(
            X[col].astype('string'),
            categories=categories
        )

    return X


def prepare_ae_features(df):
    transformed = add_temporal_features(df)

    missing = [
        col for col in numerical_features
        if col not in transformed.columns
    ]

    if missing:
        raise ValueError(
            f'Autoencoder features missing: {missing}'
        )

    X = transformed[
        numerical_features
    ].copy()

    X = X.fillna(
        ae_medians
    )

    X = X.clip(
        lower=ae_lower_clip,
        upper=ae_upper_clip,
        axis=1
    )

    if not np.isfinite(
        X.to_numpy(dtype=np.float64)
    ).all():
        raise ValueError(
            'Non-finite values in autoencoder features.'
        )

    return X


def score_dataset(df):
    X_lgb = prepare_lgb_features(df)

    raw_predictions = lgb_model.predict(
        X_lgb,
        num_iteration=lgb_model.best_iteration
    )

    if winning_model_name == 'Focal_Loss_LightGBM':
        fraud_probs = expit(
            raw_predictions
        )
    else:
        fraud_probs = np.asarray(
            raw_predictions,
            dtype=np.float64
        )

    fraud_probs = np.clip(
        fraud_probs,
        0.0,
        1.0
    )

    X_ae = prepare_ae_features(df)

    X_ae_scaled = ae_scaler.transform(
        X_ae
    ).astype(np.float32)

    tensor_x = torch.tensor(
        X_ae_scaled,
        dtype=torch.float32,
        device=device
    )

    ae_model.eval()

    with torch.no_grad():
        reconstructed = ae_model(
            tensor_x
        )

        mse_scores = torch.mean(
            (
                tensor_x
                - reconstructed
            ) ** 2,
            dim=1
        ).cpu().numpy()

    if not np.isfinite(
        fraud_probs
    ).all() or not np.isfinite(
        mse_scores
    ).all():
        raise ValueError(
            'Invalid model outputs detected.'
        )

    anomaly_flags = (
        mse_scores > ae_threshold
    )

    return (
        fraud_probs,
        mse_scores,
        anomaly_flags
    )

In [6]:
# Cell 6
def apply_direct_block_policy(
    band_mask,
    anomaly_flags
):
    return (
        band_mask
        & anomaly_flags
    )


def apply_graduated_policy(
    band_mask,
    anomaly_flags,
    probs,
    mse_scores,
    t_low,
    t_high,
    anomaly_threshold
):
    band_and_anomaly = (
        band_mask
        & anomaly_flags
    )

    severe_mse = (
        mse_scores
        > (
            GRADUATED_MSE_SEVERITY_MULTIPLIER
            * anomaly_threshold
        )
    )

    band_midpoint = (
        t_low
        + GRADUATED_RISK_MIDPOINT_FRACTION
        * (t_high - t_low)
    )

    elevated_risk = (
        probs > band_midpoint
    )

    return (
        band_and_anomaly
        & (severe_mse | elevated_risk)
    )


def route_transactions(
    probs,
    mse_scores,
    anomaly_flags,
    t_low,
    t_high,
    use_dual_model,
    escalation_policy,
    anomaly_threshold
):
    approve_mask = (
        probs < t_low
    )

    review_band_mask = (
        (probs >= t_low)
        & (probs <= t_high)
    )

    base_block_mask = (
        probs > t_high
    )

    if not use_dual_model:
        escalate_mask = np.zeros(
            len(probs),
            dtype=bool
        )

    elif escalation_policy == 'direct_block':
        escalate_mask = apply_direct_block_policy(
            review_band_mask,
            anomaly_flags
        )

    elif escalation_policy == 'graduated':
        escalate_mask = apply_graduated_policy(
            review_band_mask,
            anomaly_flags,
            probs,
            mse_scores,
            t_low,
            t_high,
            anomaly_threshold
        )

    else:
        raise ValueError(
            f'Unknown escalation policy: {escalation_policy}'
        )

    block_mask = (
        base_block_mask
        | escalate_mask
    )

    review_mask = (
        review_band_mask
        & ~escalate_mask
    )

    return (
        approve_mask,
        review_mask,
        block_mask,
        escalate_mask
    )

In [7]:
# Cell 7
def calculate_final_cost(
    amounts,
    labels,
    approve_mask,
    review_mask,
    block_mask
):
    fraud_mask = labels == 1
    legit_mask = labels == 0

    approve_fraud = (
        approve_mask & fraud_mask
    )

    block_legit = (
        block_mask & legit_mask
    )

    review_fraud = (
        review_mask & fraud_mask
    )

    review_legit = (
        review_mask & legit_mask
    )

    review_fraud_count = int(
        review_fraud.sum()
    )

    review_legit_count = int(
        review_legit.sum()
    )

    review_fraud_caught = int(
        round(
            review_fraud_count
            * review_cfg['fraud_capture_rate']
        )
    )

    review_fraud_missed = (
        review_fraud_count
        - review_fraud_caught
    )

    review_legit_approved = int(
        round(
            review_legit_count
            * review_cfg['legitimate_approval_rate']
        )
    )

    review_legit_rejected = (
        review_legit_count
        - review_legit_approved
    )

    approve_fraud_amount = float(
        amounts[approve_fraud].sum()
    )

    review_fraud_missed_amount = (
        amounts[review_fraud].sum()
        * (
            review_fraud_missed
            / review_fraud_count
        )
        if review_fraud_count > 0
        else 0.0
    )

    block_legit_amount = float(
        amounts[block_legit].sum()
    )

    review_legit_rejected_amount = (
        amounts[review_legit].sum()
        * (
            review_legit_rejected
            / review_legit_count
        )
        if review_legit_count > 0
        else 0.0
    )

    false_negative_cost = (
        approve_fraud_amount
        + int(approve_fraud.sum())
        * fin_cfg['chargeback_fee_usd']
        + review_fraud_missed_amount
        + review_fraud_missed
        * fin_cfg['chargeback_fee_usd']
    )

    false_positive_cost = (
        block_legit_amount
        * fin_cfg['profit_margin_pct']
        + int(block_legit.sum())
        * fin_cfg['false_positive_cost_usd']
        + review_legit_rejected_amount
        * fin_cfg['profit_margin_pct']
        + review_legit_rejected
        * fin_cfg['false_positive_cost_usd']
    )

    operational_review_cost = (
        int(review_mask.sum())
        * fin_cfg['manual_review_cost_usd']
    )

    total_cost = (
        false_negative_cost
        + false_positive_cost
        + operational_review_cost
    )

    return {
        'total_cost': float(total_cost),
        'false_negative_cost': float(false_negative_cost),
        'false_positive_cost': float(false_positive_cost),
        'review_operational_cost': float(
            operational_review_cost
        ),
        'auto_approved': int(
            approve_mask.sum()
        ),
        'sent_to_review': int(
            review_mask.sum()
        ),
        'auto_blocked': int(
            block_mask.sum()
        ),
        'escalated_to_block': int(
            (block_mask & ~(
                np.array(approve_mask) * False
            )).sum()
        ),
        'false_negative_count': int(
            approve_fraud.sum()
            + review_fraud_missed
        ),
        'false_positive_count': int(
            block_legit.sum()
            + review_legit_rejected
        )
    }


def calculate_baseline_costs(
    amounts,
    labels
):
    fraud_mask = labels == 1
    legit_mask = labels == 0

    approve_all = (
        amounts[fraud_mask].sum()
        + fraud_mask.sum()
        * fin_cfg['chargeback_fee_usd']
    )

    block_all = (
        amounts[legit_mask].sum()
        * fin_cfg['profit_margin_pct']
        + legit_mask.sum()
        * fin_cfg['false_positive_cost_usd']
    )

    return {
        'approve_all_cost': float(approve_all),
        'block_all_cost': float(block_all)
    }

In [8]:
# Cell 8
m6_datasets = {}

for name, path in [
    ('M6 Real', M6_REAL_PATH),
    ('M6 Stress', M6_STRESS_PATH)
]:
    df = pd.read_parquet(path)

    required_columns = {
        'TransactionAmt',
        'isFraud',
        'TransactionDT'
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f'{name} missing required columns: {sorted(missing)}'
        )

    probs, mse_scores, anomaly_flags = score_dataset(
        df
    )

    amounts = df[
        'TransactionAmt'
    ].to_numpy(dtype=np.float64)

    labels = df[
        'isFraud'
    ].to_numpy(dtype=np.int8)

    m6_datasets[name] = {
        'df': df,
        'probs': probs,
        'mse_scores': mse_scores,
        'anomaly_flags': anomaly_flags,
        'amounts': amounts,
        'labels': labels
    }

    band_mask = (
        (probs >= t_low)
        & (probs <= t_high)
    )

    print(
        f'{name}: {len(df):,} rows | '
        f'Fraud rate: {labels.mean() * 100:.3f}% | '
        f'Review band: {int(band_mask.sum()):,} | '
        f'Band anomalies: {int((band_mask & anomaly_flags).sum()):,}'
    )

C:\Users\saket\AppData\Local\Temp\ipykernel_13452\249016946.py:51: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_13452\249016946.py:51: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_13452\249016946.py:51: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_13452\249016946.py:51: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categori

M6 Real: 8,859 rows | Fraud rate: 3.093% | Review band: 2,494 | Band anomalies: 470


C:\Users\saket\AppData\Local\Temp\ipykernel_13452\249016946.py:51: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_13452\249016946.py:51: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_13452\249016946.py:51: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[col] = pd.Categorical(
C:\Users\saket\AppData\Local\Temp\ipykernel_13452\249016946.py:51: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categori

M6 Stress: 8,859 rows | Fraud rate: 3.093% | Review band: 2,494 | Band anomalies: 470


In [9]:
# Cell 9
threshold_free_results = {}

for name, data in m6_datasets.items():
    pr_auc = average_precision_score(
        data['labels'],
        data['probs']
    )

    roc_auc = roc_auc_score(
        data['labels'],
        data['probs']
    )

    threshold_free_results[name] = {
        'pr_auc': float(pr_auc),
        'roc_auc': float(roc_auc)
    }

    print(f'--- {name}: Frozen supervised model ---')
    print(f'PR-AUC:  {pr_auc:.6f}')
    print(f'ROC-AUC: {roc_auc:.6f}')

--- M6 Real: Frozen supervised model ---
PR-AUC:  0.423245
ROC-AUC: 0.853528
--- M6 Stress: Frozen supervised model ---
PR-AUC:  0.421312
ROC-AUC: 0.851425


In [10]:
# Cell 10
production_results = {}

for name, data in m6_datasets.items():
    approve_mask, review_mask, block_mask, escalate_mask = route_transactions(
        data['probs'],
        data['mse_scores'],
        data['anomaly_flags'],
        t_low,
        t_high,
        PRODUCTION_USE_DUAL_MODEL,
        PRODUCTION_ESCALATION_POLICY,
        ae_threshold
    )

    predicted_block = block_mask.astype(
        np.int8
    )

    precision = precision_score(
        data['labels'],
        predicted_block,
        zero_division=0
    )

    recall = recall_score(
        data['labels'],
        predicted_block,
        zero_division=0
    )

    f1 = f1_score(
        data['labels'],
        predicted_block,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        data['labels'],
        predicted_block
    ).ravel()

    cost = calculate_final_cost(
        data['amounts'],
        data['labels'],
        approve_mask,
        review_mask,
        block_mask
    )

    baseline = calculate_baseline_costs(
        data['amounts'],
        data['labels']
    )

    cost['approve_all_baseline_cost'] = baseline[
        'approve_all_cost'
    ]

    cost['block_all_baseline_cost'] = baseline[
        'block_all_cost'
    ]

    cost['loss_reduction_vs_approve_all_pct'] = (
        (
            baseline['approve_all_cost']
            - cost['total_cost']
        )
        / baseline['approve_all_cost']
        * 100
        if baseline['approve_all_cost'] > 0
        else 0.0
    )

    production_results[name] = {
        'routing': {
            'approve': int(approve_mask.sum()),
            'review': int(review_mask.sum()),
            'block': int(block_mask.sum()),
            'ae_escalated_to_block': int(
                escalate_mask.sum()
            )
        },
        'decision_metrics': {
            'block_precision': float(precision),
            'block_recall': float(recall),
            'block_f1': float(f1),
            'true_positives': int(tp),
            'false_positives': int(fp),
            'false_negatives': int(fn),
            'true_negatives': int(tn)
        },
        'financial_metrics': cost
    }

    print(f'--- {name}: FINAL FROZEN PRODUCTION POLICY ---')
    print(f'Approve: {approve_mask.sum():,}')
    print(f'Review:  {review_mask.sum():,}')
    print(f'Block:   {block_mask.sum():,}')
    print(f'AE escalations: {escalate_mask.sum():,}')
    print(f'Block Precision: {precision:.6f}')
    print(f'Block Recall:    {recall:.6f}')
    print(f'Block F1:        {f1:.6f}')
    print(f'Total Cost:      ${cost["total_cost"]:,.2f}')

--- M6 Real: FINAL FROZEN PRODUCTION POLICY ---
Approve: 6,230
Review:  2,133
Block:   496
AE escalations: 361
Block Precision: 0.233871
Block Recall:    0.423358
Block F1:        0.301299
Total Cost:      $27,264.17
--- M6 Stress: FINAL FROZEN PRODUCTION POLICY ---
Approve: 6,230
Review:  2,133
Block:   496
AE escalations: 361
Block Precision: 0.233871
Block Recall:    0.423358
Block F1:        0.301299
Total Cost:      $25,205.91


In [11]:
# Cell 11
ablation_configs = [
    {
        'use_dual_model': False,
        'escalation_policy': 'none'
    },
    {
        'use_dual_model': True,
        'escalation_policy': 'direct_block'
    },
    {
        'use_dual_model': True,
        'escalation_policy': 'graduated'
    }
]

ablation_rows = []

for name, data in m6_datasets.items():
    for config in ablation_configs:
        approve_mask, review_mask, block_mask, escalate_mask = route_transactions(
            data['probs'],
            data['mse_scores'],
            data['anomaly_flags'],
            t_low,
            t_high,
            config['use_dual_model'],
            config['escalation_policy'] if config['use_dual_model'] else 'graduated',
            ae_threshold
        )

        predicted_block = block_mask.astype(
            np.int8
        )

        precision = precision_score(
            data['labels'],
            predicted_block,
            zero_division=0
        )

        recall = recall_score(
            data['labels'],
            predicted_block,
            zero_division=0
        )

        cost = calculate_final_cost(
            data['amounts'],
            data['labels'],
            approve_mask,
            review_mask,
            block_mask
        )

        ablation_rows.append({
            'dataset': name,
            'use_dual_model': config['use_dual_model'],
            'escalation_policy': config['escalation_policy'],
            'block_precision': float(precision),
            'block_recall': float(recall),
            'total_cost': float(cost['total_cost']),
            'approve': int(approve_mask.sum()),
            'review': int(review_mask.sum()),
            'block': int(block_mask.sum()),
            'ae_escalations': int(escalate_mask.sum()),
            'production_config': (
                config['use_dual_model'] == PRODUCTION_USE_DUAL_MODEL
                and config['escalation_policy'] == PRODUCTION_ESCALATION_POLICY
            )
        })

ablation_df = pd.DataFrame(
    ablation_rows
)

ablation_df.to_csv(
    ABLATION_PATH,
    index=False
)

print(
    ablation_df.to_string(
        index=False
    )
)

  dataset  use_dual_model escalation_policy  block_precision  block_recall   total_cost  approve  review  block  ae_escalations  production_config
  M6 Real           False              none         0.688889      0.339416 23526.020510     6230    2494    135               0              False
  M6 Real            True      direct_block         0.209917      0.463504 28118.073274     6230    2024    605             470              False
  M6 Real            True         graduated         0.233871      0.423358 27264.166466     6230    2133    496             361               True
M6 Stress           False              none         0.688889      0.339416 21455.532462     6230    2494    135               0              False
M6 Stress            True      direct_block         0.209917      0.463504 26103.053198     6230    2024    605             470              False
M6 Stress            True         graduated         0.233871      0.423358 25205.914692     6230    2133    496       

In [12]:
# Cell 12
final_report = {
    'methodology': {
        'm6_used_only_for_final_scoring': True,
        'production_policy_fixed_before_m6_scoring': True,
        'production_use_dual_model': bool(
            PRODUCTION_USE_DUAL_MODEL
        ),
        'production_escalation_policy': (
            PRODUCTION_ESCALATION_POLICY
        ),
        'thresholds_source': FINANCIAL_POLICY_PATH,
        'supervised_model_source': supervised_model_path,
        'autoencoder_source': AE_MODEL_PATH,
        'ae_threshold': float(
            ae_threshold
        ),
        'ambiguous_band': {
            't_low': float(t_low),
            't_high': float(t_high)
        }
    },
    'threshold_free_metrics': threshold_free_results,
    'production_results': production_results,
    'ablation_results': ablation_rows
}

with open(
    FINAL_REPORT_PATH,
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        final_report,
        f,
        indent=4
    )

prediction_rows = []

for name, data in m6_datasets.items():
    approve_mask, review_mask, block_mask, escalate_mask = route_transactions(
        data['probs'],
        data['mse_scores'],
        data['anomaly_flags'],
        t_low,
        t_high,
        PRODUCTION_USE_DUAL_MODEL,
        PRODUCTION_ESCALATION_POLICY,
        ae_threshold
    )

    prediction_rows.append(
        pd.DataFrame({
            'dataset': name,
            'actual_fraud': data['labels'],
            'fraud_probability': data['probs'],
            'reconstruction_mse': data['mse_scores'],
            'ae_anomaly_flag': data['anomaly_flags'],
            'decision': np.select(
                [approve_mask, review_mask, block_mask],
                ['APPROVE', 'REVIEW', 'BLOCK'],
                default='UNKNOWN'
            ),
            'ae_escalated_to_block': escalate_mask
        })
    )

pd.concat(
    prediction_rows,
    ignore_index=True
).to_csv(
    FINAL_PREDICTIONS_PATH,
    index=False
)

print(f'Final report saved to: {FINAL_REPORT_PATH}')
print(f'Final predictions saved to: {FINAL_PREDICTIONS_PATH}')
print(f'Ablation results saved to: {ABLATION_PATH}')
print('PHASE 4 COMPLETE')

Final report saved to: m6_final_evaluation_report.json
Final predictions saved to: m6_final_predictions.csv
Ablation results saved to: m6_policy_ablation.csv
PHASE 4 COMPLETE
